In [1]:

# ============================================================
# EduInsight AI
# Phase 2: Dataset Loading and Initial Inspection
# ============================================================

from pathlib import Path
import pandas as pd
import json
import zipfile
import warnings

warnings.filterwarnings("ignore")

# 1. PROJECT PATHS

PROJECT_DIR = Path.cwd().parent
DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"

print("=" * 60)
print("EDUINSIGHT AI - DATASET LOADING")
print("=" * 60)

print("\nProject directory:", PROJECT_DIR)
print("Raw data directory:", DATA_RAW_DIR)

# 2. CHECK DATA DIRECTORY

if not DATA_RAW_DIR.exists():
    raise FileNotFoundError(
        "data/raw folder not found. Check your project structure."
    )

# 3. LIST ALL FILES

files = list(DATA_RAW_DIR.rglob("*"))

print("\nFiles found:", len([f for f in files if f.is_file()]))

for file in files:
    if file.is_file():
        print("-", file.relative_to(DATA_RAW_DIR))

# 4. EXTRACT ZIP FILES IF NEEDED

zip_files = list(DATA_RAW_DIR.glob("*.zip"))

for zip_file in zip_files:
    print(f"\nZIP archive found: {zip_file.name}")

    extract_folder = DATA_RAW_DIR / zip_file.stem

    if not extract_folder.exists():
        extract_folder.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_file, "r") as archive:
            archive.extractall(extract_folder)

        print("Extracted:", extract_folder)
    else:
        print("Already extracted:", extract_folder)

# 5. FIND DATA FILES

supported_extensions = [".csv", ".xlsx", ".xls", ".json"]

data_files = [
    file for file in DATA_RAW_DIR.rglob("*")
    if file.is_file()
    and file.suffix.lower() in supported_extensions
]

print("\nSupported data files found:", len(data_files))

for file in data_files:
    print("-", file.relative_to(DATA_RAW_DIR))

# 6. LOAD DATA FILES

loaded_datasets = {}

for file in data_files:

    try:
        extension = file.suffix.lower()

        if extension == ".csv":
            try:
                df = pd.read_csv(file, encoding="utf-8")
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding="latin-1")

        elif extension in [".xlsx", ".xls"]:
            df = pd.read_excel(file)

        elif extension == ".json":
            try:
                df = pd.read_json(file)
            except ValueError:
                with open(file, "r", encoding="utf-8") as f:
                    content = json.load(f)
                df = pd.json_normalize(content)

        loaded_datasets[str(file)] = df

        print(f"\nSuccessfully loaded: {file.name}")
        print("Shape:", df.shape)

    except Exception as error:
        print(f"\nCould not load {file.name}: {error}")

# 7. DISPLAY DATASET INFORMATION

if not loaded_datasets:
    print("\nNo supported dataset was loaded.")
    print("Check whether the ZIP file was extracted correctly.")

else:
    for file_name, df in loaded_datasets.items():

        print("\n" + "=" * 60)
        print("DATASET:", Path(file_name).name)
        print("=" * 60)

        print("\nFirst 5 rows:")
        display(df.head())

        print("\nDataset shape:")
        print(df.shape)

        print("\nColumn names:")
        print(df.columns.tolist())

        print("\nData types:")
        print(df.dtypes)

        print("\nMissing values:")
        print(df.isnull().sum())

        print("\nDuplicate rows:")
        print(df.duplicated().sum())

        print("\nStatistical information:")
        display(df.describe(include="all").T)

# 8. SAVE A SUMMARY REPORT

if loaded_datasets:

    summary = []

    for file_name, df in loaded_datasets.items():

        summary.append({
            "file_name": Path(file_name).name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "missing_values": int(df.isnull().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum())
        })

    summary_df = pd.DataFrame(summary)

    report_path = DATA_RAW_DIR / "dataset_summary.csv"
    summary_df.to_csv(report_path, index=False)

    print("\nSummary report saved to:", report_path)

print("\n" + "=" * 60)
print("DATASET INSPECTION COMPLETED")
print("=" * 60)


EDUINSIGHT AI - DATASET LOADING

Project directory: c:\Users\User\Desktop\EduInsight-AI
Raw data directory: c:\Users\User\Desktop\EduInsight-AI\data\raw

Files found: 0

Supported data files found: 0

No supported dataset was loaded.
Check whether the ZIP file was extracted correctly.

DATASET INSPECTION COMPLETED
